# Weitere Startmerkmale: Datensichtung

Wir prüfen Altersgruppe, frühere Versuche, belegte Credits und die
Angabe zu einer Behinderung zunächst auf Verfügbarkeit und Datenqualität.
Die Sichtung verwendet nur Tag 27 aus Präsentationen vor 2014J.
Eine deskriptive Gruppendifferenz ist weder eine Ursache noch bereits
ein Grund, das Merkmal zur Priorisierung zu verwenden.

In [1]:
from pathlib import Path

import pandas as pd

# Projektordner und die beiden benötigten Dateien finden.
projektordner = Path.cwd()
if projektordner.name.lower() == "notebooks":
    projektordner = projektordner.parent

wochen_datei = (
    projektordner / "data" / "processed"
    / "oulad_wochenmodellbasis_tag6_bis111.csv"
)
info_datei = projektordner / "Daten OULAD" / "studentInfo.csv"

assert wochen_datei.exists(), f"Wochenbasis fehlt: {wochen_datei}"
assert info_datei.exists(), f"studentInfo fehlt: {info_datei}"

schluessel = ["code_module", "code_presentation", "id_student"]
neue_merkmale = [
    "age_band",
    "num_of_prev_attempts",
    "studied_credits",
    "disability",
]

# Nur einen Stichtag und nur Präsentationen vor 2014J betrachten.
# So zählt jeder Kurseintrag hier einmal und 2014J steuert die
# Auswahl neuer Merkmale nicht zusätzlich.
wochen = pd.read_csv(
    wochen_datei,
    usecols=schluessel + ["stichtag", "abbruch_oder_fail"],
)
tag27 = wochen.loc[
    wochen["stichtag"].eq(27)
    & wochen["code_presentation"].ne("2014J"),
    schluessel + ["abbruch_oder_fail"],
].copy()

# Die Startmerkmale über den eindeutigen Kurseintrag ergänzen.
info = pd.read_csv(
    info_datei,
    usecols=schluessel + neue_merkmale,
)
assert not info.duplicated(schluessel).any()

sichtung = tag27.merge(
    info,
    on=schluessel,
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("Kurseinträge an Tag 27:", len(sichtung))
print("Ohne passenden studentInfo-Eintrag:",
      int(sichtung["_merge"].ne("both").sum()))

print("\nLeere Werte:")
print(sichtung[neue_merkmale].isna().sum())

print("\nFragezeichen als fehlende Angaben:")
print(sichtung[neue_merkmale].astype(str).eq("?").sum())

for merkmal in neue_merkmale:
    print(f"\n{merkmal} – häufigste Ausprägungen:")
    print(sichtung[merkmal].value_counts(dropna=False).head(10))

Kurseinträge an Tag 27: 18309
Ohne passenden studentInfo-Eintrag: 0

Leere Werte:
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
dtype: int64

Fragezeichen als fehlende Angaben:
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
dtype: int64

age_band – häufigste Ausprägungen:
age_band
0-35     12856
35-55     5347
55<=       106
Name: count, dtype: int64

num_of_prev_attempts – häufigste Ausprägungen:
num_of_prev_attempts
0    15920
1     1875
2      409
3       74
4       21
5        8
6        2
Name: count, dtype: int64

studied_credits – häufigste Ausprägungen:
studied_credits
60     9803
120    3196
30     2219
90     1686
180     436
150     386
240      92
75       88
70       78
210      76
Name: count, dtype: int64

disability – häufigste Ausprägungen:
disability
N    16554
Y     1755
Name: count, dtype: int64


## Zusatzmerkmale und spätere Ergebnisse

Die Tabellen zeigen zunächst nur beschreibende Zusammenhänge an Tag 27.
Unterschiede können auch durch die Zusammensetzung der Kurse entstehen.

In [2]:
# Die beiden Zahlenmerkmale ausdrücklich als Zahlen prüfen.
for spalte in ["num_of_prev_attempts", "studied_credits"]:
    sichtung[spalte] = pd.to_numeric(sichtung[spalte], errors="coerce")
    assert sichtung[spalte].notna().all(), f"Ungültige Werte in {spalte}"

print("Frühere Versuche:")
print(sichtung["num_of_prev_attempts"].value_counts().sort_index().to_string())

print("\nBelegte Credits:")
print(
    sichtung["studied_credits"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
)

# Für eine lesbare Tabelle fassen wir mehrere frühere Versuche zusammen.
sichtung["vorversuche_gruppe"] = pd.cut(
    sichtung["num_of_prev_attempts"],
    bins=[-1, 0, 1, float("inf")],
    labels=["0", "1", "2 oder mehr"],
)

# Diese Tabellen zeigen Fallzahlen und Zielanteile je Gruppe.
# Sie bewerten noch nicht den zusätzlichen Nutzen in einem Modell.
for merkmal in ["age_band", "vorversuche_gruppe", "disability"]:
    tabelle = (
        sichtung.groupby(merkmal, observed=True, dropna=False)[
            "abbruch_oder_fail"
        ]
        .agg(kurseintraege="size", spaetere_faelle="sum", anteil="mean")
    )
    tabelle["anteil_prozent"] = (100 * tabelle["anteil"]).round(1)
    print(f"\n{merkmal}:")
    display(
        tabelle[
            ["kurseintraege", "spaetere_faelle", "anteil_prozent"]
        ]
    )

# Bei Credits betrachten wir zunächst die Verteilung nach Ergebnis,
# ohne willkürlich eine Grenze für „viele Credits“ festzulegen.
print("\nCredits nach späterem Ergebnis:")
display(
    sichtung.groupby("abbruch_oder_fail")["studied_credits"]
    .describe()[["count", "25%", "50%", "75%"]]
)

Frühere Versuche:
num_of_prev_attempts
0    15920
1     1875
2      409
3       74
4       21
5        8
6        2

Belegte Credits:
count    18309.000000
mean        77.294773
std         39.180596
min         30.000000
10%         30.000000
25%         60.000000
50%         60.000000
75%         90.000000
90%        120.000000
max        630.000000
Name: studied_credits, dtype: float64

age_band:


,kurseintraege,spaetere_faelle,anteil_prozent
age_band,,,
0-35,12856,6174,48.0
35-55,5347,2183,40.8
55<=,106,29,27.4



vorversuche_gruppe:


,kurseintraege,spaetere_faelle,anteil_prozent
vorversuche_gruppe,,,
0,15920,6925,43.5
1,1875,1107,59.0
2 oder mehr,514,354,68.9



disability:


,kurseintraege,spaetere_faelle,anteil_prozent
disability,,,
N,16554,7433,44.9
Y,1755,953,54.3



Credits nach späterem Ergebnis:


,count,25%,50%,75%
abbruch_oder_fail,,,,
False,9923.0,60.0,60.0,90.0
True,8386.0,60.0,60.0,120.0


## Zusatznutzen im personengetrennten Modellvergleich

Alle Varianten verwenden dieselben Trainings- und Validierungsfälle.
Wir vergleichen die bisherigen Eingaben mit Vorversuchen, Credits und
Altersgruppe. Die Ergebnisse dienen der Merkmalsentscheidung innerhalb
der Entwicklungsdaten; 2014J bleibt aus diesem Vergleich heraus.

In [3]:
# Die vollständigen Wochenmerkmale für Tag 27 einlesen.
merkmale_bisher = [
    "code_module",
    "code_presentation",
    "anmeldetag",
    "assessments_faellig",
    "abgaben",
    "banked_faellige",
    "assessments_fehlend",
    "klicks_bis_stichtag",
    "klicks_letzte_7_tage",
]
zusatzmerkmale = [
    "num_of_prev_attempts",
    "studied_credits",
    "age_band",
]

vergleichsdaten = pd.read_csv(wochen_datei)
vergleichsdaten = vergleichsdaten.loc[
    vergleichsdaten["stichtag"].eq(27)
    & vergleichsdaten["code_presentation"].ne("2014J")
].copy()

# Die bereits geprüften Startmerkmale aus unserer Datensichtung ergänzen.
vergleichsdaten = vergleichsdaten.merge(
    sichtung[schluessel + zusatzmerkmale],
    on=schluessel,
    how="left",
    validate="one_to_one",
)

varianten = {
    "Bisher": merkmale_bisher,
    "+ Vorversuche": merkmale_bisher + ["num_of_prev_attempts"],
    "+ Credits": merkmale_bisher + ["studied_credits"],
    "+ Beide": merkmale_bisher
        + ["num_of_prev_attempts", "studied_credits"],
    "+ Altersgruppe": merkmale_bisher + ["age_band"],
}

assert len(vergleichsdaten) == len(sichtung)
assert not vergleichsdaten[
    merkmale_bisher + zusatzmerkmale + ["abbruch_oder_fail"]
].isna().any().any()

print("Kurseinträge:", len(vergleichsdaten))
print("Spätere Fälle:", int(vergleichsdaten["abbruch_oder_fail"].sum()))

Kurseinträge: 18309
Spätere Fälle: 8386


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def baue_zusatzmodell(merkmale):
    """Baut für jede Variante dieselbe Modellart."""
    kategorien = [
        spalte for spalte in merkmale
        if spalte in ["code_module", "code_presentation", "age_band"]
    ]
    zahlen = [spalte for spalte in merkmale if spalte not in kategorien]

    vorbereitung = ColumnTransformer([
        (
            "kategorien",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            kategorien,
        ),
        ("zahlen", StandardScaler(), zahlen),
    ])

    return Pipeline([
        ("vorbereitung", vorbereitung),
        ("modell", HistGradientBoostingClassifier(random_state=42)),
    ])


def kennzahlen_top20(validierung, risiko):
    """Berechnet Recall und Trefferquote bei 20 % Hinweisen je Kurs."""
    auswahl = validierung[
        ["code_module", "code_presentation", "abbruch_oder_fail"]
    ].copy()
    auswahl["risiko"] = risiko
    kurs = ["code_module", "code_presentation"]

    rang = auswahl.groupby(kurs)["risiko"].rank(
        ascending=False, method="first"
    )
    groesse = auswahl.groupby(kurs)["risiko"].transform("size")
    hinweis = rang.le(np.ceil(groesse * 0.20))

    treffer = auswahl.loc[hinweis, "abbruch_oder_fail"].sum()
    recall = treffer / auswahl["abbruch_oder_fail"].sum()
    trefferquote = treffer / hinweis.sum()
    return recall, trefferquote

In [5]:
ziel = vergleichsdaten["abbruch_oder_fail"].astype(int)
falten = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

ergebnisse_zusatz = []

for fold, (train_idx, valid_idx) in enumerate(
    falten.split(
        vergleichsdaten,
        ziel,
        groups=vergleichsdaten["id_student"],
    ),
    start=1,
):
    training = vergleichsdaten.iloc[train_idx]
    validierung = vergleichsdaten.iloc[valid_idx]

    # Innerhalb eines Folds sehen alle Varianten dieselben Personen.
    for name, merkmale in varianten.items():
        modell = baue_zusatzmodell(merkmale)
        modell.fit(
            training[merkmale],
            training["abbruch_oder_fail"].astype(int),
        )
        risiko = modell.predict_proba(
            validierung[merkmale]
        )[:, 1]

        recall, trefferquote = kennzahlen_top20(
            validierung, risiko
        )
        ergebnisse_zusatz.append({
            "fold": fold,
            "variante": name,
            "AP": average_precision_score(
                validierung["abbruch_oder_fail"], risiko
            ),
            "recall_20": recall,
            "trefferquote_20": trefferquote,
        })

    print(f"Fold {fold} fertig")

vergleich_zusatz = pd.DataFrame(ergebnisse_zusatz)

print("\nMittelwerte über die drei Folds:")
display(
    vergleich_zusatz.groupby("variante")[
        ["AP", "recall_20", "trefferquote_20"]
    ].mean().round(3)
)

print("\nAP in jedem einzelnen Fold:")
display(
    vergleich_zusatz.pivot(
        index="fold",
        columns="variante",
        values="AP",
    ).round(3)
)

Fold 1 fertig
Fold 2 fertig
Fold 3 fertig

Mittelwerte über die drei Folds:


,AP,recall_20,trefferquote_20
variante,,,
+ Altersgruppe,0.717,0.336,0.765
+ Beide,0.720,0.337,0.767
+ Credits,0.717,0.335,0.764
+ Vorversuche,0.720,0.337,0.769
Bisher,0.717,0.333,0.759



AP in jedem einzelnen Fold:


variante,+ Altersgruppe,+ Beide,+ Credits,+ Vorversuche,Bisher
fold,,,,,
1,0.711,0.714,0.709,0.714,0.712
2,0.716,0.717,0.716,0.718,0.716
3,0.725,0.730,0.726,0.727,0.724


## Stabilität des Merkmals „frühere Versuche“

Wir vergleichen das bisherige Modell mit derselben Variante plus
`num_of_prev_attempts` an mehreren Kurswochen. Pro Woche verwenden
beide Varianten dieselben personengetrennten Folds. 2014J bleibt
außerhalb dieser Merkmalsentscheidung.

In [6]:
stichtage_pruefung = [6, 13, 27, 55, 111]

# Das Startmerkmal gehört zum Kurseintrag und ist für alle Wochen gleich.
vorversuche = pd.read_csv(
    info_datei,
    usecols=schluessel + ["num_of_prev_attempts"],
)
assert not vorversuche.duplicated(schluessel).any()

# Nur die ausgewählten Wochen aus früheren Präsentationen verwenden.
mehrere_wochen = pd.read_csv(wochen_datei)
mehrere_wochen = mehrere_wochen.loc[
    mehrere_wochen["stichtag"].isin(stichtage_pruefung)
    & mehrere_wochen["code_presentation"].ne("2014J")
].copy()

mehrere_wochen = mehrere_wochen.merge(
    vorversuche,
    on=schluessel,
    how="left",
    validate="many_to_one",
)

mehrere_wochen["num_of_prev_attempts"] = pd.to_numeric(
    mehrere_wochen["num_of_prev_attempts"],
    errors="coerce",
)
assert mehrere_wochen[
    merkmale_bisher + ["num_of_prev_attempts", "abbruch_oder_fail"]
].notna().all().all()

print("Kurseinträge je Stichtag:")
print(mehrere_wochen["stichtag"].value_counts().sort_index())

Kurseinträge je Stichtag:
stichtag
6      18934
13     18724
27     18309
55     17632
111    16607
Name: count, dtype: int64


In [7]:
varianten_wochen = {
    "Bisher": merkmale_bisher,
    "+ Vorversuche": merkmale_bisher + ["num_of_prev_attempts"],
}

ergebnisse_wochen = []

for tag in stichtage_pruefung:
    daten = mehrere_wochen.loc[
        mehrere_wochen["stichtag"].eq(tag)
    ].reset_index(drop=True)
    ziel = daten["abbruch_oder_fail"].astype(int)

    falten = StratifiedGroupKFold(
        n_splits=3,
        shuffle=True,
        random_state=42,
    )

    for fold, (train_idx, valid_idx) in enumerate(
        falten.split(daten, ziel, groups=daten["id_student"]),
        start=1,
    ):
        training = daten.iloc[train_idx]
        validierung = daten.iloc[valid_idx]

        for name, merkmale in varianten_wochen.items():
            modell = baue_zusatzmodell(merkmale)
            modell.fit(
                training[merkmale],
                training["abbruch_oder_fail"].astype(int),
            )
            risiko = modell.predict_proba(
                validierung[merkmale]
            )[:, 1]

            recall, trefferquote = kennzahlen_top20(
                validierung, risiko
            )
            ergebnisse_wochen.append({
                "stichtag": tag,
                "fold": fold,
                "variante": name,
                "AP": average_precision_score(
                    validierung["abbruch_oder_fail"], risiko
                ),
                "recall_20": recall,
                "trefferquote_20": trefferquote,
            })

    print(f"Tag {tag} fertig")

ergebnisse_wochen = pd.DataFrame(ergebnisse_wochen)

print("\nMittelwerte je Woche:")
display(
    ergebnisse_wochen.groupby(["stichtag", "variante"])[
        ["AP", "recall_20", "trefferquote_20"]
    ].mean().round(3)
)

print("\nAP-Unterschied je Fold (+ Vorversuche minus Bisher):")
ap_vergleich = ergebnisse_wochen.pivot(
    index=["stichtag", "fold"],
    columns="variante",
    values="AP",
)
ap_vergleich["Unterschied"] = (
    ap_vergleich["+ Vorversuche"] - ap_vergleich["Bisher"]
)
display(ap_vergleich.round(3))

Tag 6 fertig
Tag 13 fertig
Tag 27 fertig
Tag 55 fertig
Tag 111 fertig

Mittelwerte je Woche:


AP  recall_20  trefferquote_20
stichtag variante                                        
6        + Vorversuche  0.650      0.287            0.682
         Bisher         0.644      0.286            0.679
13       + Vorversuche  0.679      0.311            0.727
         Bisher         0.675      0.307            0.719
27       + Vorversuche  0.720      0.337            0.769
         Bisher         0.717      0.333            0.759
55       + Vorversuche  0.764      0.385            0.837
         Bisher         0.763      0.383            0.833
111      + Vorversuche  0.826      0.455            0.911
         Bisher         0.825      0.454            0.909


AP-Unterschied je Fold (+ Vorversuche minus Bisher):


variante       + Vorversuche  Bisher  Unterschied
stichtag fold                                    
6        1             0.660   0.654        0.005
         2             0.648   0.643        0.005
         3             0.641   0.634        0.007
13       1             0.687   0.684        0.004
         2             0.669   0.667        0.003
         3             0.679   0.675        0.004
27       1             0.714   0.712        0.002
         2             0.718   0.716        0.003
         3             0.727   0.724        0.003
55       1             0.762   0.761        0.001
         2             0.759   0.760       -0.000
         3             0.770   0.768        0.002
111      1             0.826   0.825        0.002
         2             0.820   0.821       -0.000
         3             0.830   0.829        0.001

## Entscheidung für den Prototyp

Frühere Modulversuche hängen deskriptiv deutlich mit dem späteren
Ergebnis zusammen. Zusätzlich zu Kurskontext, VLE-Aktivität und
Assessments verbessern sie die personengetrennte Modellbewertung
jedoch nur geringfügig. An Tag 6 steigt die AP von 0,644 auf 0,650,
der Recall bei 20 % Hinweisen von 28,6 % auf 28,7 %.

Credits und Altersgruppe liefern an Tag 27 keinen nennenswerten
Zusatznutzen. Die Angabe zu einer Behinderung wurde nicht als
Modellmerkmal getestet. Die vorhandenen Wochenmodelle bleiben daher
für den vorführbaren Prototyp bestehen. Diese Merkmalsentscheidungen
sind explorativ; 2014J ist kein unberührter Abschlusstest.